In [453]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd

verbose = False

In [454]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


In [ ]:
def learnQ(targets, covariates,embedding_dim,verbose):

    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)


    for step in range(800):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        # use l2 norm to regularize Q
        # regularization is needed for the EBM data because we want the synthetic covariates
        # to be in a reasonable range.
        lambda_l1_YQ = 0 # trying penalization of size of synthetic covariates YQ
        lambda_l2_Q = 1
        lambda_l2_w = 10 # large penalty otherwise this overfits to the EBM data

        l1_YQ = sum([torch.sum(abs(YQ)) for YQ in YQ_list]) # 
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w) + (lambda_l1_YQ * l1_YQ)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        scheduler.step()

        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")
            print(f"Step {step:4d} | Loss: {loss.item():.8f} | w: {w_sol.detach().numpy().round(3)}")
            print(f"Grad norm: {Q.grad.norm().item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final


# Synthetic Data Experiment

In [ ]:
# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_timepoints, n_outcomes, n_units)  # The data now has shape - (n_timepoints, n_outcomes, n_units)


test = Y[len(Y)-1] # take the last time point - Has shape (outcomes x units) (K,N)
train = Y[0:len(Y)-1] # take all covariate matrices up until the last time point - for each time point has shape (outcomes x units) (T-1, K, N)

verbose=True
if (verbose):
    print("Test data: \n\n", test)
    print("\nTrain data: \n\n", train)
    # print(train.shape) #  It is (timepoints-1, outcomes, units)

# split the data into the target vectors and the covariate matrices
# goal here is to use final outcomes to predict final target
train_target_vectors = [torch.from_numpy(train[i][0:train[i].shape[0],0:1]).squeeze() for i in range(len(train))]
train_covariate_matrices = [torch.from_numpy(train[i][0:train[i].shape[0],1:train[i].shape[1]]) for i in range(len(train))]

verbose=False
if verbose == True:
    for i in range(len(train)):
        print("\nTarget vector", i, ":\n", train_target_vectors[i])
        print("\nCovariate matrix", i, ":\n", train_covariate_matrices[i])


# The treated unit at the final time point
test_target_vector = torch.from_numpy(test[0:test.shape[0],0:1]).squeeze()
test_covariate_matrix = torch.from_numpy(test[0:test.shape[0],1:test.shape[1]])

# run the learn Q function:
# NB: embedding into a much lower dimension also works! (try 2)
# embedding dimension doesn't make that much of a difference interestingly
Q_final, w_final = learnQ(train_target_vectors, train_covariate_matrices, 3, False)

torch.set_printoptions(sci_mode=False, precision=2)
print("\n train original covariates:\n", train_covariate_matrices)

print("\n test [Original] covariates: \n", test_covariate_matrix)
print("\n test [Synthetic] covariates: \n",test_covariate_matrix@Q_final)
print("\nFinal weights: \n", w_final.round(2))
print("\nComputed: \n", test_covariate_matrix@Q_final@w_final)
print("\nGoal: \n", test_target_vector)
print("Final Q: \n", Q_final.round(2))

if verbose == True:
    for i in range(len(train_target_vectors)): 
        print("Final Q: \n", Q_final)
        print("\nTraining Sample ", i)
        print("Goal: \n", train_target_vectors[i])
        print("Computed \n", train_covariate_matrices[i] @ Q_final @ w_final)
        print("\n")



# EBM Factor Model Experiment

### Example of how the data is being re-shaped into the form required by learnQ function

In [671]:
np.random.seed(215)

T = 4
K = 2
N_minus_1 = 2

print("Original EBM data:\n")
# T x K
out_treated = np.random.rand(T, K)
print("out_treated:\n", out_treated)

# ((N-1)*T) x K  — this is what EBM's code produces
out_control = np.random.rand(N_minus_1 * T, K)
print("\nout_control:\n", out_control)

# Reshape control to (T, N-1, K) then transpose to (T, K, N-1)
out_control_3d = out_control.reshape(T, N_minus_1, K).transpose(0, 2, 1)  

# Add unit dimension to treated and concatenate
out_trt_3d = out_treated[:, np.newaxis, :].transpose(0, 2, 1)  

Y = np.concatenate([out_trt_3d, out_control_3d], axis=2)  

print("\nFinal Reshaped data:\n",Y)

Original EBM data:

out_treated:
 [[0.15392159 0.48956512]
 [0.18198889 0.20908939]
 [0.22316718 0.08422693]
 [0.91593932 0.60770076]]

out_control:
 [[0.30851461 0.74596553]
 [0.81033323 0.39261118]
 [0.41442981 0.17074698]
 [0.76798541 0.40117711]
 [0.3622311  0.40013476]
 [0.28704036 0.0098913 ]
 [0.33164009 0.22417675]
 [0.46732337 0.56413055]]

Final Reshaped data:
 [[[0.15392159 0.30851461 0.81033323]
  [0.48956512 0.74596553 0.39261118]]

 [[0.18198889 0.41442981 0.76798541]
  [0.20908939 0.17074698 0.40117711]]

 [[0.22316718 0.3622311  0.28704036]
  [0.08422693 0.40013476 0.0098913 ]]

 [[0.91593932 0.33164009 0.46732337]
  [0.60770076 0.22417675 0.56413055]]]


Now we actually reshape the data produced by EBM's code:

In [718]:
treated_data = pd.read_csv('treated_data.csv')
control_data = pd.read_csv('control_data.csv')

# Reshape the data appropriately from EBM's code.
T = treated_data.shape[0]        # 40
K = treated_data.shape[1]        # 10
N_minus_1 = control_data.shape[0] // T  # 1960 // 40 = 49

out_trt = treated_data.to_numpy()      # (40, 10)
out_control = control_data.to_numpy()  # (1960, 10)

# Reshape control to (T, N-1, K) then transpose to (T, K, N-1)
out_control_3d = out_control.reshape(T, N_minus_1, K).transpose(0, 2, 1)  # (40, 10, 49)

# Add unit dimension to treated and concatenate
out_trt_3d = out_trt[:, np.newaxis, :].transpose(0, 2, 1)  

Y = np.concatenate([out_trt_3d, out_control_3d], axis=2)  

# Train/test split
test = Y[len(Y)-1] 
train = Y[0:len(Y)-1]


Run the learnQ function on the training data:

In [747]:
train_target_vectors = [torch.from_numpy(train[i][0:train[i].shape[0],0:1]).squeeze() for i in range(len(train))]
train_covariate_matrices = [torch.from_numpy(train[i][0:train[i].shape[0],1:train[i].shape[1]]) for i in range(len(train))]

Q_final, w_final = learnQ(train_target_vectors, train_covariate_matrices, 10, False)

print(w_final.round(2))

Step    0 | Loss: 570.00382120
Step    0 | Loss: 570.00382120 | w: [-0.     0.     0.     0.051  0.233  0.082  0.43   0.     0.09   0.114]
Grad norm: 144.81831814
Step  200 | Loss: 434.87920801
Step  200 | Loss: 434.87920801 | w: [ 0.     0.001  0.149  0.     0.14   0.041  0.439 -0.     0.032  0.199]
Grad norm: 5.78268281
Step  400 | Loss: 427.31832439
Step  400 | Loss: 427.31832439 | w: [0.    0.    0.091 0.012 0.026 0.025 0.638 0.    0.06  0.147]
Grad norm: 1.43430105
Step  600 | Loss: 426.97314589
Step  600 | Loss: 426.97314589 | w: [ 0.037  0.032 -0.     0.03  -0.     0.005  0.722  0.003  0.067  0.104]
Grad norm: 1.63248103
[-0.    0.04  0.02  0.03  0.    0.01  0.72 -0.    0.07  0.11]


In [745]:
test_target_vector = torch.from_numpy(test[0:test.shape[0],0:1]).squeeze()
test_covariate_matrix = torch.from_numpy(test[0:test.shape[0],1:test.shape[1]])
pred_learnQ = test_covariate_matrix.numpy()@ Q_final @ w_final  # (K,)


print("Synthetic Covariates:\n",(test_covariate_matrix.numpy()@ Q_final).round(1))
print("final learnQ weights:\n", w_final.round(2))
print("final Q matrix:\n", Q_final.round(2))

print("test_target_vector:\n", test_target_vector)
print("pred_learnQ:\n", pred_learnQ.round(2))



w_avg = pd.read_csv('T_0=40_K=10_w_avg.csv').to_numpy()
# is test_covariate matrix the correct thing to apply w_avg to?
pred_avg = test_covariate_matrix @ w_avg
print("pred_avg:\n", pred_avg.numpy().T.round(2))


# VALIDATE THIS 
# our methods are producing remarkably similar predictions.
# something to note is that the model_t1 data is the data at time T+1, and is noiseless. This is how EBM evaluates his model.
# need to think about how to make apples to apples comparison. 
# I will have to compare my predictions also to the model_t1
# this means making a prediction based on the noiseless model_t1 data.
# have to investigate how EBM calculates the bias.

# Is it possible my method works better for data with fewer units?

Synthetic Covariates:
 [[ 0.   3.   3.2  2.7 -0.2  1.9  7.8 -0.   2.7  3. ]
 [ 0.   2.4  2.6  2.2 -0.1  1.6  6.3  0.   2.3  2.4]
 [ 0.   2.7  2.9  2.5 -0.2  1.6  6.9  0.   2.5  2.5]
 [ 0.   3.8  3.9  3.4 -0.2  2.3  9.5  0.   3.4  3.6]
 [ 0.   3.1  3.3  2.8 -0.2  1.9  8.   0.   2.8  3.1]
 [ 0.   3.4  3.5  3.1 -0.2  2.1  8.7  0.   3.1  3.3]
 [ 0.   3.   3.2  2.8 -0.2  1.9  7.8  0.   2.8  2.9]
 [ 0.   3.7  3.9  3.4 -0.2  2.3  9.6  0.   3.4  3.6]
 [ 0.   3.   3.2  2.7 -0.1  1.8  7.6  0.   2.7  2.9]
 [ 0.   3.6  3.8  3.3 -0.2  2.2  9.3  0.   3.4  3.6]]
final learnQ weights:
 [-0.    0.12  0.14  0.11 -0.    0.09  0.31 -0.    0.12  0.12]
final Q matrix:
 [[ 0.    0.19  0.17  0.17  0.    0.11  0.45  0.    0.17  0.16]
 [-0.    0.19  0.2   0.16 -0.    0.12  0.46  0.    0.18  0.18]
 [-0.    0.23  0.26  0.21 -0.05  0.15  0.64 -0.    0.22  0.23]
 [ 0.    0.04  0.07  0.06  0.    0.03  0.15  0.    0.06  0.04]
 [ 0.01  0.06  0.06  0.06  0.    0.03  0.13  0.    0.02  0.05]
 [ 0.    0.01  0.    0.02  0.

In [ ]:
# load weights and post-treatment data
w_sep = pd.read_csv('T_0=40_K=10_w_sep.csv').to_numpy()
w_cat = pd.read_csv('T_0=40_K=10_w_cat.csv').to_numpy()
w_avg = pd.read_csv('T_0=40_K=10_w_avg.csv').to_numpy()
model_t1 = pd.read_csv('T_0=40_K=10_model_t1.csv').to_numpy()

# split into treated and control
model_t1_treated = model_t1[0, :]    # (K,) — first row is treated unit
model_t1_control = model_t1[1:, :]   # (N-1, K) — remaining rows are controls

# predictions from each method
pred_sep = model_t1_control.T @ w_sep    # (K,)
pred_cat = model_t1_control.T @ w_cat
pred_avg = model_t1_control.T @ w_avg
pred_learnQ = test_covariate_matrix.numpy()@ Q_final @ w_final  # (K,)

# compare against truth
print("True treated outcome:  ", model_t1_treated.round(3))
print("learnQ prediction:     ", pred_learnQ.round(3))
print("SCM sep prediction:    ", pred_sep.round(3))
print("SCM cat prediction:    ", pred_cat.round(3))
print("SCM avg prediction:    ", pred_avg.round(3))

True treated outcome:   [4.556 4.556 4.556 4.556 4.556 4.556 4.556 4.556 4.556 4.556]
learnQ prediction:      [4.609 4.737 4.493 5.096 4.615 3.929 4.246 4.921 5.599 4.766]
SCM sep prediction:     [[4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]
 [4.146]]
SCM cat prediction:     [[4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]
 [4.269]]
SCM avg prediction:     [[4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]
 [4.538]]


## Toy Example

In [ ]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 1.0], dtype=torch.float64)
Y_2 = torch.tensor([[9.0, 9.0], [2.0, 56.0]], dtype=torch.float64)

d_3 = torch.tensor([2.0, 7.0], dtype=torch.float64)
Y_3 = torch.tensor([[2.0, 1.0], [4.0, 5.0]], dtype=torch.float64)

d_4 = torch.tensor([8.0, 7.0], dtype=torch.float64)
Y_4 = torch.tensor([[6.0, 7.0], [1.0, 9.0]], dtype=torch.float64)




target_vectors = [d_1, d_2, d_3, d_4]
covariate_matrices = [Y_1, Y_2, Y_3, Y_4]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 6, False)


Step    0 | Loss: 88.36956387
Step  200 | Loss: 83.89083664
Step  400 | Loss: 83.68999669
Step  600 | Loss: 83.54118857
Step  800 | Loss: 83.53940439
Step 1000 | Loss: 83.53974022
Step 1200 | Loss: 83.53925808
Step 1400 | Loss: 83.53930585
Step 1600 | Loss: 83.53883250
Step 1800 | Loss: 83.53901438


In [458]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0], [4.0, 6.0]], dtype=torch.float64)

d_3 = torch.tensor([4.0, 4.0], dtype=torch.float64)
Y_3 = torch.tensor([[1.0, 3.0], [2.0, 4.0]], dtype=torch.float64)

target_vectors = [d_1, d_2, d_3]
covariate_matrices = [Y_1,Y_2,Y_3]

In [459]:
print(target_vectors[0].shape)   # should be (2,)
print(covariate_matrices[0].shape)  # should be (2, 2)

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 3, False)


print(w_final.round(2))
print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(d_3)
print(Y_3@Q_final@w_final)

print("Synthetic Donor 1: \n", Y_1@Q_final)
print("Synthetic Donor 2: \n", Y_2@Q_final)
print("Synthetic Donor 2: \n", Y_3@Q_final)

torch.Size([2])
torch.Size([2, 2])
Step    0 | Loss: 33.71912495
Step  200 | Loss: 19.86590159
Step  400 | Loss: 17.86754195
Step  600 | Loss: 17.72074313
Step  800 | Loss: 17.69807881
Step 1000 | Loss: 17.68938107
Step 1200 | Loss: 17.68719305
Step 1400 | Loss: 17.68599609
Step 1600 | Loss: 17.68548070
Step 1800 | Loss: 17.68525016
[-0. -0.  1.]
tensor([6., 2.], dtype=torch.float64)
tensor([4.27, 3.67], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([2.13, 3.07], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([4.86, 4.27], dtype=torch.float64)
Synthetic Donor 1: 
 tensor([[-0.00,  0.20,  4.27],
        [-0.00,  0.25,  3.67]], dtype=torch.float64)
Synthetic Donor 2: 
 tensor([[-0.00,  0.10,  2.13],
        [-0.00,  0.31,  3.07]], dtype=torch.float64)
Synthetic Donor 2: 
 tensor([[-0.00,  0.15,  4.86],
        [-0.00,  0.20,  4.27]], dtype=torch.float64)


In [460]:
# Data More donors than time points.
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(w_final.round(2))




Step    0 | Loss: 74.09506824
Step  200 | Loss: 1.68906200
Step  400 | Loss: 1.14762923
Step  600 | Loss: 0.99151832
Step  800 | Loss: 0.95811428
Step 1000 | Loss: 0.94108430
Step 1200 | Loss: 0.93349423
Step 1400 | Loss: 0.92581964
Step 1600 | Loss: 0.91995205
Step 1800 | Loss: 0.91576849
tensor([6., 2.], dtype=torch.float64)
tensor([5.78, 1.84], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([5.17, 3.20], dtype=torch.float64)
[ 0.44 -0.    0.29  0.03  0.24]


## Psuedo Inverses

### single training example

In [461]:
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

target_vectors = [d_1]
covariate_matrices = [Y_1]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 4, False)

# make sense that these would be the same, because in the last step of the 
# algorithm, we learn the optimal w for the given Q.
# and this is given by the psuedo-inverse
# maybe this means we don't have to have one of the differentiation
# steps - can just use the pseudo-inverse instead of gradient descent on w.
pinv = torch.linalg.pinv(Y_1@ Q_final)
print("\noptimal w using psuedo inverse: \n", pinv@d_1)
print("optimal w using algorithm: \n", w_final.round(2))

Step    0 | Loss: 184.58965332
Step  200 | Loss: 0.75188195
Step  400 | Loss: 0.65810330
Step  600 | Loss: 0.64516704
Step  800 | Loss: 0.64113552
Step 1000 | Loss: 0.63839181
Step 1200 | Loss: 0.63697609
Step 1400 | Loss: 0.63517967
Step 1600 | Loss: 0.63370632
Step 1800 | Loss: 0.63263990

optimal w using psuedo inverse: 
 tensor([0.17, 0.36, 0.33, 0.13], dtype=torch.float64)
optimal w using algorithm: 
 [0.17 0.35 0.34 0.14]


### Multiple training examples

In [462]:
# Data More donors than time points.
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)

# very similar - but does this hold inside the loop?
print("mean of psuedo-inverses: \n", ((torch.linalg.pinv(Y_1@Q_final) @ d_1) + (torch.linalg.pinv(Y_2@Q_final) @ d_2)) / 2)

print(w_final.round(2))

Step    0 | Loss: 74.09506824
Step  200 | Loss: 1.68906200
Step  400 | Loss: 1.14762923
Step  600 | Loss: 0.99151832
Step  800 | Loss: 0.95811428
Step 1000 | Loss: 0.94108430
Step 1200 | Loss: 0.93349423
Step 1400 | Loss: 0.92581964
Step 1600 | Loss: 0.91995205
Step 1800 | Loss: 0.91576849
mean of psuedo-inverses: 
 tensor([ 0.43, -0.00,  0.27,  0.07,  0.19], dtype=torch.float64)
[ 0.44 -0.    0.29  0.03  0.24]


# Modified function to check if the psuedo inverse is being used

In [463]:

def learnQcheck(targets, covariates,embedding_dim,verbose):
    # CHECK
    Q_list = []
    w_list = []

    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)


    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        if step % 1 == 0:
            Q_list.append(Q.detach().numpy().copy())
            w_list.append(w_sol.detach().numpy().copy())

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.1
        lambda_l2_w = 1
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        scheduler.step()

        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final, Q_list, w_list


In [464]:
# Data More donors than time points.
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final, Q_list, w_list = learnQcheck(target_vectors, covariate_matrices, 5, False)

Step    0 | Loss: 74.09506824
Step  200 | Loss: 1.68906200
Step  400 | Loss: 1.14762923
Step  600 | Loss: 0.99151832
Step  800 | Loss: 0.95811428
Step 1000 | Loss: 0.94108430
Step 1200 | Loss: 0.93349423
Step 1400 | Loss: 0.92581964
Step 1600 | Loss: 0.91995205
Step 1800 | Loss: 0.91576849


In [465]:

# What would happen if I just used w_sol as the mean of the psuedo inverses at each step, instead?
# would the mean of the pseudo inverses still sum to 1?

print(Q_list[0].round(2))
print(Q_list[1].round(2))
print(Q_list[len(Q_list)-1].round(2))

for i in range(len(Q_list)):
    print("\nw at step ", i*10, ":\n", w_list[i].round(2))
    
    print("mean of psuedo-inverses: \n", ((torch.linalg.pinv(Y_1@Q_list[i]) @ d_1) + (torch.linalg.pinv(Y_2@Q_list[i]) @ d_2)) / 2)




[[ 0.38 -0.11 -1.46  0.63 -0.44]
 [ 0.47  0.28  0.79 -1.26 -1.71]
 [-0.08  0.44  0.5   0.14  0.3 ]
 [-0.45 -1.82 -1.88  0.35 -0.73]]
[[ 0.37 -0.12 -1.47  0.64 -0.43]
 [ 0.46  0.27  0.78 -1.25 -1.7 ]
 [-0.09  0.43  0.49  0.15  0.31]
 [-0.46 -1.83 -1.89  0.36 -0.72]]
[[-0.21  0.   -0.12  0.72 -0.12]
 [ 0.77  0.    0.47 -0.73  0.35]
 [-0.84  0.   -0.48 -0.06 -0.36]
 [ 0.67 -0.    0.39  0.73  0.3 ]]

w at step  0 :
 [-0.  0.  0.  1.  0.]
mean of psuedo-inverses: 
 tensor([ 0.43, -0.46, -0.23, -0.76,  0.39], dtype=torch.float64)

w at step  10 :
 [ 0. -0. -0.  1.  0.]
mean of psuedo-inverses: 
 tensor([ 0.47, -0.41, -0.19, -0.79,  0.27], dtype=torch.float64)

w at step  20 :
 [-0. -0. -0.  1.  0.]
mean of psuedo-inverses: 
 tensor([ 0.50, -0.35, -0.15, -0.79,  0.16], dtype=torch.float64)

w at step  30 :
 [0. 0. 0. 1. 0.]
mean of psuedo-inverses: 
 tensor([ 0.50, -0.30, -0.12, -0.77,  0.05], dtype=torch.float64)

w at step  40 :
 [ 0.  0. -0.  1. -0.]
mean of psuedo-inverses: 
 tensor([ 0.4

In [466]:

def learnQpinv(targets, covariates,embedding_dim,verbose):
    # CHECK
    Q_list = []
    w_list = []

    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)


    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        pinvs = [torch.linalg.pinv(Y @ Q) @ target_vectors for (Y, target_vectors) in zip(covariate_matrices, target_vectors)]
        w_sol = torch.mean(torch.stack(pinvs), dim=0)

        if step % 1 == 0:
            Q_list.append(Q.detach().numpy().copy())
            w_list.append(w_sol.detach().numpy().copy())

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.1
        
        l2_Q = torch.sum(Q**2)
    
        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        scheduler.step()

        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final, Q_list, w_list


In [467]:
# Data More donors than time points.
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final, Q_list, w_list = learnQpinv(target_vectors, covariate_matrices, 5, False)

Step    0 | Loss: 4.21228002
Step  200 | Loss: 1.02639971
Step  400 | Loss: 0.59266597
Step  600 | Loss: 0.42799302
Step  800 | Loss: 1.30132943
Step 1000 | Loss: 0.33393604
Step 1200 | Loss: 0.31449639
Step 1400 | Loss: 0.29359551
Step 1600 | Loss: 0.27699494
Step 1800 | Loss: 0.26492586
